In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Read image
img = cv2.imread('Chapter1/images/bluescreen_tree.png')

# Convert BGR to RGB for display
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Function to show the image
def show(img, title="Image"):
    plt.figure(figsize=(5,5))
    if len(img.shape) == 2:  
        # grayscale image
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

1. Sobel Edge Detector

The Sobel operator computes gradients in the x and y directions.

Sobel X

In [ ]:
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

In [ ]:
sobel_x = cv2.Sobel(
    gray,
    cv2.CV_64F,
    1, 0,
    ksize=3
)

sobel_x = cv2.convertScaleAbs(sobel_x)

show(sobel_x, "Sobel X")

Sobel Y

In [ ]:
sobel_y = cv2.Sobel(
    gray,
    cv2.CV_64F,
    0, 1,
    ksize=3
)

sobel_y = cv2.convertScaleAbs(sobel_y)

show(sobel_y, "Sobel Y")

Combined Sobel Magnitude

In [ ]:
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

magnitude = np.uint8(
    magnitude / magnitude.max() * 255
)

show(magnitude, "Sobel Edge Detector")

2. Prewitt Edge Detector

OpenCV does not provide a built-in Prewitt operator, so we use convolution kernels.

Prewitt Kernels

In [ ]:
prewitt_x_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=np.float32)

prewitt_y_kernel = np.array([
    [ 1,  1,  1],
    [ 0,  0,  0],
    [-1, -1, -1]
], dtype=np.float32)

Prewitt X

In [ ]:
prewitt_x = cv2.filter2D(
    gray,
    cv2.CV_64F,
    prewitt_x_kernel
)

prewitt_x = cv2.convertScaleAbs(prewitt_x)

show(prewitt_x, "Prewitt X")

Prewitt Y

In [ ]:
prewitt_y = cv2.filter2D(
    gray,
    cv2.CV_64F,
    prewitt_y_kernel
)

prewitt_y = cv2.convertScaleAbs(prewitt_y)

show(prewitt_y, "Prewitt Y")

Combined Prewitt Magnitude

In [ ]:
gx = cv2.filter2D(
    gray,
    cv2.CV_64F,
    prewitt_x_kernel
)

gy = cv2.filter2D(
    gray,
    cv2.CV_64F,
    prewitt_y_kernel
)

magnitude = np.sqrt(gx**2 + gy**2)

magnitude = np.uint8(
    magnitude / magnitude.max() * 255
)

show(magnitude, "Prewitt Edge Detector")

3. Canny Edge Detector (OpenCV)

The Canny algorithm consists of:

- a. Gaussian smoothing
- b. Gradient computation
- c. Non-Maximum Suppression
- d. Double Thresholding
- e. Edge Tracking by Hysteresis

OpenCV performs all steps internally.

In [ ]:
edges = cv2.Canny(
    gray,
    threshold1=100,
    threshold2=200
)

show(edges, "Canny Edge Detector")

4. Canny Edge Detector (From Scratch)

This implementation demonstrates the main steps manually.

In [ ]:
blurred = cv2.GaussianBlur(
    gray,
    (5,5),
    1.4
)

show(blurred, "Gaussian Smoothed")

Step 2: Gradient Computation (Sobel)

In [ ]:
gx = cv2.Sobel(
    blurred,
    cv2.CV_64F,
    1, 0,
    ksize=3
)

gy = cv2.Sobel(
    blurred,
    cv2.CV_64F,
    0, 1,
    ksize=3
)

magnitude = np.sqrt(gx**2 + gy**2)

direction = np.arctan2(gy, gx)

show(
    np.uint8(magnitude / magnitude.max() * 255),
    "Gradient Magnitude"
)

Step 3: Non-Maximum Suppression

In [ ]:
def non_maximum_suppression(magnitude, direction):
    
    rows, cols = magnitude.shape
    
    output = np.zeros((rows, cols), dtype=np.float32)

    angle = direction * 180 / np.pi
    angle[angle < 0] += 180

    for i in range(1, rows-1):
        for j in range(1, cols-1):

            q = 255
            r = 255

            # 0 degrees
            if (0 <= angle[i,j] < 22.5) or (157.5 <= angle[i,j] <= 180):
                q = magnitude[i, j+1]
                r = magnitude[i, j-1]

            # 45 degrees
            elif (22.5 <= angle[i,j] < 67.5):
                q = magnitude[i+1, j-1]
                r = magnitude[i-1, j+1]

            # 90 degrees
            elif (67.5 <= angle[i,j] < 112.5):
                q = magnitude[i+1, j]
                r = magnitude[i-1, j]

            # 135 degrees
            elif (112.5 <= angle[i,j] < 157.5):
                q = magnitude[i-1, j-1]
                r = magnitude[i+1, j+1]

            if magnitude[i,j] >= q and magnitude[i,j] >= r:
                output[i,j] = magnitude[i,j]

    return output

nms = non_maximum_suppression(
    magnitude,
    direction
)

show(
    np.uint8(nms / nms.max() * 255),
    "Non-Maximum Suppression"
)

Step 4: Double Thresholding

In [ ]:
def double_threshold(img,
                     low=50,
                     high=100):

    strong = 255
    weak = 75

    result = np.zeros_like(img, dtype=np.uint8)

    strong_i, strong_j = np.where(img >= high)
    weak_i, weak_j = np.where(
        (img >= low) &
        (img < high)
    )

    result[strong_i, strong_j] = strong
    result[weak_i, weak_j] = weak

    return result

thresholded = double_threshold(
    nms,
    50,
    100
)

show(
    thresholded,
    "Double Threshold"
)

Step 5: Edge Tracking by Hysteresis

In [ ]:
def hysteresis(img):

    rows, cols = img.shape

    strong = 255
    weak = 75

    result = img.copy()

    for i in range(1, rows-1):
        for j in range(1, cols-1):

            if result[i,j] == weak:

                neighborhood = result[
                    i-1:i+2,
                    j-1:j+2
                ]

                if np.any(neighborhood == strong):
                    result[i,j] = strong
                else:
                    result[i,j] = 0

    return result

final_edges = hysteresis(
    thresholded
)

show(
    final_edges,
    "Canny From Scratch"
)

### Combined comparision

In [ ]:
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1)

sobel = np.sqrt(
    sobel_x**2 +
    sobel_y**2
)

sobel = np.uint8(
    sobel / sobel.max() * 255
)

gx = cv2.filter2D(gray, cv2.CV_64F, prewitt_x_kernel)
gy = cv2.filter2D(gray, cv2.CV_64F, prewitt_y_kernel)

prewitt = np.sqrt(
    gx**2 +
    gy**2
)

prewitt = np.uint8(
    prewitt / prewitt.max() * 255
)

canny = cv2.Canny(gray, 100, 200)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(sobel, cmap='gray')
plt.title("Sobel")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(prewitt, cmap='gray')
plt.title("Prewitt")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(canny, cmap='gray')
plt.title("Canny")
plt.axis('off')

plt.tight_layout()
plt.show()